# **Audio Data Preparation: leaning, Noise Reduction, Augmentation, Dataset Curation**
This Colab notebook demonstrates core techniques from Module 4.                             
**Sections:**
- 1) Setup
- 2) Load Audio
- 3) Cleaning (trim, normalize, filter)
- 4) Noise Reduction (spectral & deep denoiser)
- 5) Augmentation
- 6) Simple Dataset Curation

In [2]:
# @title 1) Setup
!pip install -U denoiser --quiet
!pip install torch torchaudio matplotlib --quiet
!pip install librosa noisereduce denoiser soundfile torch torchaudio scipy --quiet

import librosa
import numpy as np
import soundfile as sf
import torchaudio
from torchaudio.utils import download_asset
import matplotlib.pyplot as plt
from IPython import display as disp
import torch
import torchaudio
from denoiser import pretrained
from denoiser.dsp import convert_audio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.8/49.8 kB 2.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 122.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# @title 2) Load Audio
!wget https://facebookresearch.github.io/denoiser/audio/noisy/alex_noisy.mp3

y, sr = librosa.load(librosa.ex("trumpet"), sr=16000)
print(f"Loaded audio: {len(y)/sr:.2f}s at {sr}Hz")
disp.display(disp.Audio(y, rate=sr))

--2025-07-07 19:25:37--  https://facebookresearch.github.io/denoiser/audio/noisy/alex_noisy.mp3
Resolving facebookresearch.github.io (facebookresearch.github.io)... 185.199.108.153, 185.199.109.153, 185.199.110.153, ...
Connecting to facebookresearch.github.io (facebookresearch.github.io)|185.199.108.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 339840 (332K) [audio/mp3]
Saving to: ‘alex_noisy.mp3’

alex_noisy.mp3      100%[===================>] 331.88K  --.-KB/s    in 0.02s   

2025-07-07 19:25:37 (14.6 MB/s) - ‘alex_noisy.mp3’ saved [339840/339840]



Loaded audio: 5.33s at 16000Hz


In [4]:
# @title 3) Cleaning
## 3.1 Trim silence
y_trim, _ = librosa.effects.trim(y, top_db=20)
print(f"Trimmed length: {len(y_trim)/sr:.2f}s")
disp.display(disp.Audio(y_trim, rate=sr))

Trimmed length: 3.04s


In [5]:
## 3.2 Peak normalization
y_norm = y_trim / np.max(np.abs(y_trim)) * 0.9
disp.display(disp.Audio(y_norm, rate=sr))

In [6]:
## 3.3 High-pass filter (remove DC and rumble)
from scipy.signal import butter, filtfilt
b, a = butter(4, 80, btype='high', fs=sr)
y_filt = filtfilt(b, a, y_norm)
disp.display(disp.Audio(y_filt, rate=sr))

In [7]:
# @title 4) Noise reduction
import noisereduce as nr
# 4.1 Spectral subtraction (noisereduce)
y_specden = nr.reduce_noise(y=y_filt, sr=sr)
disp.display(disp.Audio(y_specden, rate=sr))

In [8]:
# 4.2 Deep denoiser (Facebook denoiser)
# import torch
# from denoiser.pretrained import Denoiser

# dn = Denoiser()  # loads pretrained denoiser

# # Pad the audio to a multiple of 256
# audio_tensor = torch.from_numpy(y_filt).float().unsqueeze(0)
# padding = (256 - audio_tensor.shape[-1] % 256) % 256
# audio_tensor = torch.nn.functional.pad(audio_tensor, (0, padding)).float() # Ensure float32

# # Move the audio tensor to the same device as the model
# audio_tensor = audio_tensor.to(dn.device)

# dn_out = dn(audio_tensor)
# y_dn = dn_out.squeeze().numpy()
# # Remove padding
# y_dn = y_dn[:len(y_filt)]
# disp.display(disp.Audio(y_dn, rate=sr))

In [9]:
# 5) Augmentation
augmented = {}
# 5.1 Additive noise
noise = np.random.randn(len(y_filt)) * 0.02
augmented['noise+'] = y_filt + noise
disp.display(disp.Audio(augmented['noise+'], rate=sr))

In [10]:
# 5.2 Time stretching
augmented['slow'] = librosa.effects.time_stretch(y_filt, rate=0.9)
augmented['fast'] = librosa.effects.time_stretch(y_filt, rate=1.1)
disp.display(disp.Audio(augmented['slow'], rate=sr))
disp.display(disp.Audio(augmented['fast'], rate=sr))

In [11]:
# 5.3 Pitch shifting
augmented['pitch_up'] = librosa.effects.pitch_shift(y_filt, sr=sr, n_steps=2)
augmented['pitch_down'] = librosa.effects.pitch_shift(y_filt, sr=sr, n_steps=-2)
disp.display(disp.Audio(augmented['pitch_up'], rate=sr))
disp.display(disp.Audio(augmented['pitch_down'], rate=sr))

In [12]:
# 5.4 Gain perturbation
augmented['louder'] = y_filt * 1.2
augmented['quieter'] = y_filt * 0.8
disp.display(disp.Audio(augmented['louder'], rate=sr))
disp.display(disp.Audio(augmented['quieter'], rate=sr))

In [13]:
# 5.5 Reverberation (simple echo)
ir = np.zeros(int(0.3*sr)); ir[0] = 1; ir[int(0.1*sr)] = 0.6
augmented['reverb'] = np.convolve(y_filt, ir, mode='full')[:len(y_filt)]
disp.display(disp.Audio(augmented['reverb'], rate=sr))

In [14]:
# 6. Simple dataset curation demo
files = ['orig'] + list(augmented.keys())
dataset = {name: None for name in files}
# store arrays (in practice, save to disk)
dataset['orig'] = y_filt
for name, arr in augmented.items():
  dataset[name] = arr
print("Dataset versions:", list(dataset.keys()))
print("Samples per version:", {k: len(v) for k, v in dataset.items() if v is not None})

dataset[name] = arr
print("Dataset versions:", list(dataset.keys()))
print("Samples per version:", {k: len(v) for k, v in dataset.items() if v is not None})


Dataset versions: ['orig', 'noise+', 'slow', 'fast', 'pitch_up', 'pitch_down', 'louder', 'quieter', 'reverb']
Samples per version: {'orig': 48640, 'noise+': 48640, 'slow': 54044, 'fast': 44218, 'pitch_up': 48640, 'pitch_down': 48640, 'louder': 48640, 'quieter': 48640, 'reverb': 48640}
Dataset versions: ['orig', 'noise+', 'slow', 'fast', 'pitch_up', 'pitch_down', 'louder', 'quieter', 'reverb']
Samples per version: {'orig': 48640, 'noise+': 48640, 'slow': 54044, 'fast': 44218, 'pitch_up': 48640, 'pitch_down': 48640, 'louder': 48640, 'quieter': 48640, 'reverb': 48640}


# Task


## Load `alex noisy.mp3`

Load the `alex_noisy.mp3` audio file using librosa, ensuring the correct sampling rate.


In [15]:
y_alex, sr_alex = librosa.load('alex_noisy.mp3', sr=sr)
print(f"Loaded audio: {len(y_alex)/sr_alex:.2f}s at {sr_alex}Hz")
disp.display(disp.Audio(y_alex, rate=sr_alex))

Loaded audio: 16.87s at 16000Hz


## Apply cleaning

Apply the same cleaning steps (trimming silence, peak normalization, high-pass filtering) to the `alex_noisy.mp3` audio.


In [16]:
# 1. Trim silence
y_alex_trim, _ = librosa.effects.trim(y_alex, top_db=20)
print(f"Trimmed length: {len(y_alex_trim)/sr_alex:.2f}s")
disp.display(disp.Audio(y_alex_trim, rate=sr_alex))

# 2. Peak normalization
y_alex_norm = y_alex_trim / np.max(np.abs(y_alex_trim)) * 0.9
disp.display(disp.Audio(y_alex_norm, rate=sr_alex))

# 3. High-pass filter
from scipy.signal import butter, filtfilt
b, a = butter(4, 80, btype='high', fs=sr_alex)
y_alex_filt = filtfilt(b, a, y_alex_norm)
disp.display(disp.Audio(y_alex_filt, rate=sr_alex))

Output hidden; open in https://colab.research.google.com to view.

## Apply noise reduction

Apply both spectral subtraction and deep denoiser to the cleaned `alex_noisy.mp3` audio.


In [17]:
# 4.1 Spectral subtraction (noisereduce)
y_alex_specden = nr.reduce_noise(y=y_alex_filt, sr=sr_alex)
disp.display(disp.Audio(y_alex_specden, rate=sr_alex))

# 4.2 Deep denoiser (Facebook denoiser)
# Import Denoiser from denoiser.pretrained is not working, try importing from denoiser.model
# Corrected import and model loading
from denoiser import pretrained

dn = pretrained.dns48(pretrained=True)  # loads pretrained denoiser

# Convert to tensor, add batch dimension, and ensure float32
audio_tensor_alex = torch.from_numpy(y_alex_filt.copy()).float().unsqueeze(0)

# Pad the audio to a multiple of 256
padding = (256 - audio_tensor_alex.shape[-1] % 256) % 256
audio_tensor_alex_padded = torch.nn.functional.pad(audio_tensor_alex, (0, padding)).float() # Ensure float32

# Determine the device and move the model and tensor
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dn.to(device)
audio_tensor_alex_padded = audio_tensor_alex_padded.to(device)


# Apply the denoiser
dn_out_alex = dn(audio_tensor_alex_padded)

# Remove padding and convert back to numpy
# Detach the tensor before converting to numpy
y_alex_dn = dn_out_alex.squeeze().cpu().detach().numpy()
y_alex_dn = y_alex_dn[:len(y_alex_filt)]

disp.display(disp.Audio(y_alex_dn, rate=sr_alex))

Downloading: "https://dl.fbaipublicfiles.com/adiyoss/denoiser/dns48-11decc9d8e3f0998.th" to /root/.cache/torch/hub/checkpoints/dns48-11decc9d8e3f0998.th
100%|██████████| 72.0M/72.0M [00:00<00:00, 114MB/s]


In [18]:
# 4.1 Spectral subtraction (noisereduce)

y_alex_specden = nr.reduce_noise(y=y_alex_filt, sr=sr_alex)
disp.display(disp.Audio(y_alex_specden, rate=sr_alex))

# 4.2 Deep denoiser (Facebook denoiser)

from denoiser import pretrained

dn = pretrained.dns48(pretrained=True)  # loads pretrained denoiser

# Convert to tensor, add batch dimension, and ensure float32
audio_tensor_alex = torch.from_numpy(y_alex_filt.copy()).float().unsqueeze(0)

# Determine the device and move the model and tensor
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dn.to(device)
audio_tensor_alex = audio_tensor_alex.to(device)

# Pad the audio to a multiple of 256
padding = (256 - audio_tensor_alex.shape[-1] % 256) % 256
audio_tensor_alex_padded = torch.nn.functional.pad(audio_tensor_alex, (0, padding)).float() # Ensure float32


# Apply the denoiser
dn_out_alex = dn(audio_tensor_alex_padded)

# Remove padding and convert back to numpy
# Detach the tensor before converting to numpy
y_alex_dn = dn_out_alex.squeeze().cpu().detach().numpy()
y_alex_dn = y_alex_dn[:len(y_alex_filt)]

disp.display(disp.Audio(y_alex_dn, rate=sr_alex))

In [19]:
# 4.2 Deep denoiser (Facebook denoiser)
# Attempting to load the model using pretrained.dns48
dn = pretrained.dns48(pretrained=True)  # loads pretrained denoiser

# Determine the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dn.to(device) # Move model to device

# Convert to tensor, add batch dimension, and ensure float32
# Create a contiguous copy of the numpy array before converting to tensor
audio_tensor_alex = torch.from_numpy(y_alex_filt.copy()).float().unsqueeze(0)

# Pad the audio to a multiple of 256
padding = (256 - audio_tensor_alex.shape[-1] % 256) % 256
audio_tensor_alex_padded = torch.nn.functional.pad(audio_tensor_alex, (0, padding)).float() # Ensure float32

# Move the audio tensor to the same device as the model
audio_tensor_alex_padded = audio_tensor_alex_padded.to(device)

# Apply the denoiser
dn_out_alex = dn(audio_tensor_alex_padded)

# Remove padding and convert back to numpy
# Detach the tensor before converting to numpy
y_alex_dn = dn_out_alex.squeeze().cpu().detach().numpy()
y_alex_dn = y_alex_dn[:len(y_alex_filt)]

disp.display(disp.Audio(y_alex_dn, rate=sr_alex))

In [20]:
# 4.2 Deep denoiser (Facebook denoiser)
# Attempting to load the model using pretrained.dns48
dn = pretrained.dns48(pretrained=True)  # loads pretrained denoiser

# Determine the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dn.to(device) # Move model to device

# Convert to tensor, add batch dimension, and ensure float32
# Create a contiguous copy of the numpy array before converting to tensor
audio_tensor_alex = torch.from_numpy(y_alex_filt.copy()).float().unsqueeze(0)

# Pad the audio to a multiple of 256
padding = (256 - audio_tensor_alex.shape[-1] % 256) % 256
audio_tensor_alex_padded = torch.nn.functional.pad(audio_tensor_alex, (0, padding)).float() # Ensure float32

# Move the audio tensor to the same device as the model
audio_tensor_alex_padded = audio_tensor_alex_padded.to(device)

# Apply the denoiser
dn_out_alex = dn(audio_tensor_alex_padded)

# Remove padding and convert back to numpy
# Detach the tensor before converting to numpy
y_alex_dn = dn_out_alex.squeeze().cpu().detach().numpy()
y_alex_dn = y_alex_dn[:len(y_alex_filt)]

disp.display(disp.Audio(y_alex_dn, rate=sr_alex))

In [21]:
# 4.2 Deep denoiser (Facebook denoiser)
# Attempting to load the model using pretrained.dns48
dn = pretrained.dns48(pretrained=True)  # loads pretrained denoiser

# Determine the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dn.to(device) # Move model to device

# Convert to tensor, add batch dimension, and ensure float32
# Create a contiguous copy of the numpy array before converting to tensor
audio_tensor_alex = torch.from_numpy(y_alex_filt.copy()).float().unsqueeze(0)

# Pad the audio to a multiple of 256
padding = (256 - audio_tensor_alex.shape[-1] % 256) % 256
audio_tensor_alex_padded = torch.nn.functional.pad(audio_tensor_alex, (0, padding)).float() # Ensure float32

# Move the audio tensor to the same device as the model
audio_tensor_alex_padded = audio_tensor_alex_padded.to(device)

# Apply the denoiser
dn_out_alex = dn(audio_tensor_alex_padded)

# Remove padding and convert back to numpy
# Detach the tensor before converting to numpy
y_alex_dn = dn_out_alex.squeeze().cpu().detach().numpy()
y_alex_dn = y_alex_dn[:len(y_alex_filt)]

disp.display(disp.Audio(y_alex_dn, rate=sr_alex))

In [22]:
# 4.2 Deep denoiser (Facebook denoiser)
# Attempting to load the model using pretrained.dns48
dn = pretrained.dns48(pretrained=True)  # loads pretrained denoiser

# Determine the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dn.to(device) # Move model to device

# Convert to tensor, add batch dimension, and ensure float32
# Create a contiguous copy of the numpy array before converting to tensor
audio_tensor_alex = torch.from_numpy(y_alex_filt.copy()).float().unsqueeze(0)

# Pad the audio to a multiple of 256
padding = (256 - audio_tensor_alex.shape[-1] % 256) % 256
audio_tensor_alex_padded = torch.nn.functional.pad(audio_tensor_alex, (0, padding)).float() # Ensure float32

# Move the audio tensor to the same device as the model
audio_tensor_alex_padded = audio_tensor_alex_padded.to(device)

# Apply the denoiser
dn_out_alex = dn(audio_tensor_alex_padded)

# Remove padding and convert back to numpy
# Detach the tensor before converting to numpy
y_alex_dn = dn_out_alex.squeeze().cpu().detach().numpy()
y_alex_dn = y_alex_dn[:len(y_alex_filt)]

disp.display(disp.Audio(y_alex_dn, rate=sr_alex))

## Apply augmentation


Apply the same augmentation techniques (additive noise, time stretching, pitch shifting, gain perturbation, reverberation, and SpecAugment) to the noise-reduced `alex_noisy.mp3` audio.


In [23]:
# 1. Create an empty dictionary
augmented_alex = {}

# 2. Additive noise
noise_alex = np.random.randn(len(y_alex_dn)) * 0.02
augmented_alex['noise+'] = y_alex_dn + noise_alex
print("Additive Noise:")
disp.display(disp.Audio(augmented_alex['noise+'], rate=sr_alex))

# 3. Time stretching
augmented_alex['slow'] = librosa.effects.time_stretch(y_alex_dn, rate=0.9)
augmented_alex['fast'] = librosa.effects.time_stretch(y_alex_dn, rate=1.1)
print("Time Stretching (Slow):")
disp.display(disp.Audio(augmented_alex['slow'], rate=sr_alex))
print("Time Stretching (Fast):")
disp.display(disp.Audio(augmented_alex['fast'], rate=sr_alex))

# 4. Pitch shifting
augmented_alex['pitch_up'] = librosa.effects.pitch_shift(y_alex_dn, sr=sr_alex, n_steps=2)
augmented_alex['pitch_down'] = librosa.effects.pitch_shift(y_alex_dn, sr=sr_alex, n_steps=-2)
print("Pitch Shifting (Up):")
disp.display(disp.Audio(augmented_alex['pitch_up'], rate=sr_alex))
print("Pitch Shifting (Down):")
disp.display(disp.Audio(augmented_alex['pitch_down'], rate=sr_alex))

# 5. Gain perturbation
augmented_alex['louder'] = y_alex_dn * 1.2
augmented_alex['quieter'] = y_alex_dn * 0.8
print("Gain Perturbation (Louder):")
disp.display(disp.Audio(augmented_alex['louder'], rate=sr_alex))
print("Gain Perturbation (Quieter):")
disp.display(disp.Audio(augmented_alex['quieter'], rate=sr_alex))

# 6. Reverberation (simple echo)
ir_alex = np.zeros(int(0.3*sr_alex)); ir_alex[0] = 1; ir_alex[int(0.1*sr_alex)] = 0.6
augmented_alex['reverb'] = np.convolve(y_alex_dn, ir_alex, mode='full')[:len(y_alex_dn)]
print("Reverberation:")
disp.display(disp.Audio(augmented_alex['reverb'], rate=sr_alex))

# 7. SpecAugment on log-mel (just masking example)
S_alex = librosa.feature.melspectrogram(y=y_alex_dn, sr=sr_alex, n_mels=64)
S_db_alex = librosa.power_to_db(S_alex, ref=np.max)
# mask some time and freq bands
S_db_alex_masked = S_db_alex.copy()
freq_mask_alex =  slice(10, 20)
time_mask_alex = slice(30, 50)
S_db_alex_masked[freq_mask_alex, :] = S_db_alex_masked.min()
S_db_alex_masked[:, time_mask_alex] = S_db_alex_masked.min()
augmented_alex['specaug'] = S_db_alex_masked
print("SpecAugment applied to spectrogram.")

Output hidden; open in https://colab.research.google.com to view.

## Curate dataset

### Subtask:
Create a simple dataset dictionary containing the original and augmented versions of the `alex_noisy.mp3` audio.


**Reasoning**:
Create a dataset dictionary for the alex_noisy audio, including the original cleaned and noise-reduced version and all augmented versions, and then print the keys and lengths of the items in the dictionary.



In [24]:
# 1. Create a list of keys for the dataset dictionary
files_alex = ['orig'] + list(augmented_alex.keys())

# 2. Initialize an empty dictionary named dataset_alex
dataset_alex = {}

# 3. Add the original cleaned and noise-reduced audio data (y_alex_dn)
dataset_alex['orig'] = y_alex_dn

# 4. Iterate through the augmented_alex dictionary and add each augmented audio version
for name, arr in augmented_alex.items():
  dataset_alex[name] = arr

# 5. Print the keys of the dataset_alex dictionary
print("Dataset versions for alex_noisy:", list(dataset_alex.keys()))

# 6. Print the number of samples per version. Handle 'specaug' separately.
print("Samples per version for alex_noisy:")
for k, v in dataset_alex.items():
    if k == 'specaug':
        print(f"{k}: {v.shape}")
    else:
        print(f"{k}: {len(v)}")

Dataset versions for alex_noisy: ['orig', 'noise+', 'slow', 'fast', 'pitch_up', 'pitch_down', 'louder', 'quieter', 'reverb', 'specaug']
Samples per version for alex_noisy:
orig: 268800
noise+: 268800
slow: 298667
fast: 244364
pitch_up: 268800
pitch_down: 268800
louder: 268800
quieter: 268800
reverb: 268800
specaug: (64, 526)
